In [1]:
"""
First spin up vLLM server with:

vllm serve deepseek-ai/DeepSeek-V2-Lite-Chat \
  --dtype auto \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.85 \
  --max-num-seqs 16 \
  --tensor-parallel-size 2 --data-parallel-size 1 --enable-expert-parallel --enable-eplb
  
Can also profile with:

vllm bench serve \
  --backend vllm \
  --model deepseek-ai/DeepSeek-V2-Lite-Chat \
  --dataset-name random \
  --num-prompts 50 \
  --request-rate 10 \
  --result-filename metrics.json
  
"""

'\nFirst spin up vLLM server with:\n\nvllm serve Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4   --quantization gptq_marlin   --dtype auto   --max-model-len 4096   --gpu-memory-utilization 0.85   --max-num-seqs 16   --tensor-parallel-size 2 --data-parallel-size 1 --enable-expert-parallel --enable-eplb\n  \nCan also profile with:\n\nvllm bench serve   --backend vllm   --model Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4   --dataset-name random   --num-prompts 50   --request-rate 10   --result-filename metrics.json\n  \n'

In [2]:
import os
# Force vLLM to use spawn method to avoid CUDA fork errors in Jupyter
#os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Tell the C++ compiler's linker exactly where to find Conda's CUDA libraries
#conda_prefix = os.environ.get("CONDA_PREFIX", "/home/dylan/miniconda3")
#os.environ["LIBRARY_PATH"] = f"{conda_prefix}/lib:" + os.environ.get("LIBRARY_PATH", "")

In [3]:
from experiments_vllm import *
from data import *
from transformers import AutoTokenizer

/home/dylan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [4]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [5]:
# Configuration
# For full dataset: n_samples = 15000, max_new_tokens = 100, batch_size = 16
model = "Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4"
port = 8000
n_samples = 1000
max_new_tokens = 100
max_model_len = 4096
gpu_memory_utilization = 0.85
n_gpus = 2
enable_expert_parallel = True
batch_size = 16
n_warmup_samples = 5

In [6]:
# Select prompts
sample_prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a python script to scrape a website.",
    "What are the benefits of MoE (Mixture of Experts) architectures?",
    "Tell me a short sci-fi story about a sentient coffee machine.",
    "Summarize the history of the Roman Empire in 3 paragraphs."
]

# Use MMLU questions
dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
mmlu_prompts = format_prompts_mmlu(dataset)

Streaming cais/mmlu (all) (samples: 1000)...


In [7]:
# Run experiment over generalized MMLU questions
overall_results, avg_tpot = await run_experiment_vllm_throughput(model,
                                                         mmlu_prompts,
                                                         seed=seed,
                                                         max_new_tokens=max_new_tokens,
                                                         max_model_len=max_model_len,
                                                         gpu_memory_utilization=gpu_memory_utilization,
                                                         n_gpus=n_gpus,
                                                         n_warmup_samples=n_warmup_samples,
                                                         print_output=False,
                                                         enable_expert_parallel=enable_expert_parallel)

Starting vLLM server for Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4...
Waiting for server to initialize ...
(APIServer pid=1004412) INFO 04-30 14:55:19 [utils.py:299] 
(APIServer pid=1004412) INFO 04-30 14:55:19 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=1004412) INFO 04-30 14:55:19 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.20.0
(APIServer pid=1004412) INFO 04-30 14:55:19 [utils.py:299]   █▄█▀ █     █     █     █  model   Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4
(APIServer pid=1004412) INFO 04-30 14:55:19 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=1004412) INFO 04-30 14:55:19 [utils.py:299] 
(APIServer pid=1004412) INFO 04-30 14:55:19 [utils.py:233] non-default args: {'model_tag': 'Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', 'model': 'Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', 'seed': 43, 'max_model_len': 4096, 'quantization': 'gptq_marlin', 'override_generation_config': {'temperature': 0.0}, 'tensor_parallel_size': 2, 'enable_expert_parallel': True, 'gpu_memory

(APIServer pid=1004412) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(APIServer pid=1004412) INFO 04-30 14:55:20 [model.py:555] Resolved architecture: Qwen2MoeForCausalLM
(APIServer pid=1004412) INFO 04-30 14:55:20 [model.py:1680] Using max model len 4096
(APIServer pid=1004412) INFO 04-30 14:55:21 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
(APIServer pid=1004412) WARNING 04-30 14:55:21 [nixl_utils.py:34] NIXL is not available
(APIServer pid=1004412) WARNING 04-30 14:55:21 [nixl_utils.py:44] NIXL agent config is not available
(APIServer pid=1004412) INFO 04-30 14:55:21 [gptq_marlin.py:235] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
(APIServer pid=1004412) INFO 04-30 14:55:21 [vllm.py:840] Asynchronous scheduling is enabled.
(APIServer pid=1004412) INFO 04-30 14:55:21 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
WARNING 04-30 14:55:31 [nixl_utils.py:34] NIXL is not available
WARNIN

(Worker_TP1_EP1 pid=1004529) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(Worker_TP0_EP0 pid=1004528) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(Worker_TP0_EP0 pid=1004528) INFO 04-30 14:55:42 [weight_utils.py:904] Filesystem type for checkpoints: EXT4. Checkpoint size: 7.84 GiB. Available RAM: 86.99 GiB.
(Worker_TP0_EP0 pid=1004528) INFO 04-30 14:55:42 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (EXT4) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


...

(EngineCore pid=1004473) ERROR 04-30 14:56:27 [multiproc_executor.py:283] Worker proc VllmWorker-0 died unexpectedly, shutting down executor.


(EngineCore pid=1004473) Process EngineCore:
(EngineCore pid=1004473) Traceback (most recent call last):
(EngineCore pid=1004473)   File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=1004473)     self.run()
(EngineCore pid=1004473)   File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore pid=1004473)     self._target(*self._args, **self._kwargs)
(EngineCore pid=1004473)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1140, in run_engine_core
(EngineCore pid=1004473)     raise e
(EngineCore pid=1004473)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=1004473)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=1004473)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=1004473)     return func(*a

(APIServer pid=1004412) Traceback (most recent call last):
(APIServer pid=1004412)   File "/home/dylan/.local/bin/vllm", line 6, in <module>
(APIServer pid=1004412)     sys.exit(main())
(APIServer pid=1004412)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/entrypoints/cli/main.py", line 92, in main
(APIServer pid=1004412)     args.dispatch_function(args)
(APIServer pid=1004412)   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/entrypoints/cli/serve.py", line 122, in cmd
(APIServer pid=1004412)     uvloop.run(run_server(args))
(APIServer pid=1004412)   File "/home/dylan/.local/lib/python3.10/site-packages/uvloop/__init__.py", line 69, in run
(APIServer pid=1004412)     return loop.run_until_complete(wrapper())
(APIServer pid=1004412)   File "uvloop/loop.pyx", line 1518, in uvloop.loop.Loop.run_until_complete
(APIServer pid=1004412)   File "/home/dylan/.local/lib/python3.10/site-packages/uvloop/__init__.py", line 48, in wrapper
(APIServer pid=1004412)     return 

An error occurred during inference: vLLM server process terminated unexpectedly.


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
# Run experiment over specific MMLU subjects
n_subjects = 10
subject_results = {}
for subject in MMLU_SUBJECTS[:n_subjects]:
    dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed, subset=subject)
    mmlu_prompts = format_prompts_mmlu(dataset)
    
    # Repeat prompts to get to n_samples
    while len(mmlu_prompts) < n_samples:
        mmlu_prompts += mmlu_prompts
    mmlu_prompts = mmlu_prompts[:n_samples]
    
    results, avg_tpot = await run_experiment_vllm_throughput(model,
                                                         mmlu_prompts,
                                                         seed=seed,
                                                         max_new_tokens=max_new_tokens,
                                                         max_model_len=max_model_len,
                                                         gpu_memory_utilization=gpu_memory_utilization,
                                                         n_gpus=n_gpus,
                                                         print_output=False,
                                                         enable_expert_parallel=enable_expert_parallel)
    
    subject_results[subject] = results


In [ ]:
# Plot
plot_hist_tpots(overall_results, subject_results)

In [ ]:
# Run experiment for exact same prompt n_samples times
n_repeat_prompts = 10
repeat_results = {}
for i in range(n_repeat_prompts):
    repeat_prompts = [mmlu_prompts[i]] * n_samples
    results, avg_tpot = await run_experiment_vllm_throughput(model,
                                                             repeat_prompts,
                                                             seed=seed,
                                                             max_new_tokens=max_new_tokens,
                                                             max_model_len=max_model_len,
                                                             gpu_memory_utilization=gpu_memory_utilization,
                                                             n_gpus=n_gpus,
                                                             n_warmup_samples=n_warmup_samples,
                                                             print_output=False,
                                                             enable_expert_parallel=enable_expert_parallel)
    repeat_results[f"repeat prompt {i}"] = results

In [ ]:
plot_hist_tpots(overall_results, repeat_results)

In [ ]:
import pickle
with open('./vllm_benchmarking/overall_results_gpu2_expertp.pkl', 'wb') as file:
    pickle.dump(overall_results, file)
with open('./vllm_benchmarking/subject_results_gpu2_expertp.pkl', 'wb') as file:
    pickle.dump(subject_results, file)
with open('./vllm_benchmarking/repeat_results_gpu2_expertp.pkl', 'wb') as file:
    pickle.dump(repeat_results, file)